# Sales Rep Analytics — Data Cleaning & Merge Pipeline

**Inputs:** `employee_info.csv` · `mileage.csv` · `rep_perf.csv` · `sales_data.csv`  
**Output:** `merged_rep_analysis.csv` + `merged_rep_analysis.xlsx`  
**Grain of final dataset:** one row per rep per quarter

## 0. Setup & File Paths

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Edit these to match your actual file locations ──────────────────────────
EMPLOYEE_INFO_FILE = "employee_info.csv"
MILEAGE_FILE       = "mileage.csv"
REP_PERF_FILE      = "rep_perf.csv"
SALES_FILE         = "sales_data.csv"
OUTPUT_CSV         = "merged_rep_analysis.csv"
OUTPUT_XLSX        = "merged_rep_analysis.xlsx"

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Setup complete')

## Helper Functions

In [ ]:
def load(path, label):
    """Load CSV or Excel, strip column whitespace, show shape."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {label} at: {path}")
    df = pd.read_excel(path) if path.endswith(('.xlsx', '.xls')) else pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"Loaded {label}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df


def normalize_yr_qtr(series):
    """
    Normalize all quarter strings to 'YYYY QN' format.
    Handles: '2024 Q1', '2024Q1', 'Q1 2024', '2024-Q1', '1Q2024'
    """
    s = series.astype(str).str.strip()
    s = s.str.replace(r'(\d{4})[-]?Q([1-4])', r'\1 Q\2', regex=True)
    s = s.str.replace(r'Q([1-4]) (\d{4})', r'\2 Q\1', regex=True)
    s = s.str.replace(r'^([1-4])Q(\d{4})$', r'\2 Q\1', regex=True)
    return s


def attainment_tier(pct):
    """Bucket quota attainment % into a readable label."""
    if pd.isna(pct):  return 'Unknown'
    if pct >= 120:    return 'Overachiever (≥120%)'
    if pct >= 100:    return 'At Quota (100–119%)'
    if pct >= 80:     return 'Near Quota (80–99%)'
    if pct >= 50:     return 'Below Quota (50–79%)'
    return 'Far Below Quota (<50%)'

print('Helpers defined')

---
## 1. Employee Info
Columns: `Status Classification`, `Employee Number`, `Original Hire Date`, `Termination Date`, `Position`

In [ ]:
emp = load(EMPLOYEE_INFO_FILE, 'Employee Info')
emp.head(3)

In [ ]:
emp['Employee Number']    = emp['Employee Number'].astype(str).str.strip()
emp['Original Hire Date'] = pd.to_datetime(emp['Original Hire Date'], errors='coerce')
emp['Termination Date']   = pd.to_datetime(emp['Termination Date'],   errors='coerce')

# Is Active: no termination date = still employed
emp['Is Active'] = emp['Termination Date'].isna()

# Tenure: hire → termination date (or today if still active)
today = pd.Timestamp.today().normalize()
emp['Tenure Years'] = (
    (emp['Termination Date'].fillna(today) - emp['Original Hire Date']).dt.days / 365.25
).round(2)

# One row per rep (dedup by employee number)
emp_clean = emp[[
    'Employee Number', 'Status Classification', 'Position',
    'Original Hire Date', 'Termination Date', 'Is Active', 'Tenure Years'
]].drop_duplicates(subset='Employee Number')

print(f"Unique reps: {len(emp_clean):,}")
print(f"Active: {emp_clean['Is Active'].sum()} | Terminated: {(~emp_clean['Is Active']).sum()}")
emp_clean.head()

---
## 2. Mileage Data
Columns: `Bus Miles`, `Date`, `Employee Number`, `Position`  
→ Aggregated to **rep × quarter** level before merging

In [ ]:
mil = load(MILEAGE_FILE, 'Mileage')
mil.head(3)

In [ ]:
mil['Employee Number'] = mil['Employee Number'].astype(str).str.strip()
mil['Date']            = pd.to_datetime(mil['Date'], errors='coerce')
mil['Bus Miles']       = pd.to_numeric(mil['Bus Miles'], errors='coerce').fillna(0)

# Drop rows missing date or employee
mil = mil.dropna(subset=['Date', 'Employee Number'])

# Build Yr Qtr from date
mil['Yr Qtr'] = mil['Date'].dt.year.astype(str) + ' Q' + mil['Date'].dt.quarter.astype(str)

# Aggregate: sum miles + count trips per rep per quarter
mil_agg = (
    mil.groupby(['Employee Number', 'Yr Qtr'])
    .agg(
        Total_Miles   = ('Bus Miles', 'sum'),
        Mileage_Trips = ('Bus Miles', 'count'),
    )
    .reset_index()
)

print(f"Aggregated: {len(mil_agg):,} employee-quarter rows")
mil_agg.head()

---
## 3. Rep Performance by Quarter
Columns: `Sales`, `Division`, `DivisionName`, `TerritoryName`, `Yr Qtr`, `Territory`, `Goal`, `Employee Number`, `Position`

In [ ]:
rp = load(REP_PERF_FILE, 'Rep Performance')
rp.head(3)

In [ ]:
rp['Employee Number'] = rp['Employee Number'].astype(str).str.strip()
rp['Sales']           = pd.to_numeric(rp['Sales'], errors='coerce')
rp['Goal']            = pd.to_numeric(rp['Goal'],  errors='coerce')
rp['Yr Qtr']          = normalize_yr_qtr(rp['Yr Qtr'])

# ── Calculated fields ─────────────────────────────────────────────────────
rp['Quota Attainment %'] = (rp['Sales'] / rp['Goal'] * 100).round(2)
rp['Gap to Quota $']     = (rp['Sales'] - rp['Goal']).round(2)   # negative = missed
rp['Attainment Tier']    = rp['Quota Attainment %'].apply(attainment_tier)
rp['Hit Quota']          = rp['Quota Attainment %'] >= 100

rp_clean = rp[[
    'Employee Number', 'Yr Qtr',
    'Division', 'DivisionName', 'Territory', 'TerritoryName', 'Position',
    'Goal', 'Sales',
    'Quota Attainment %', 'Gap to Quota $', 'Attainment Tier', 'Hit Quota'
]]

print(f"Reps: {rp_clean['Employee Number'].nunique():,}")
print(f"Quarters: {sorted(rp_clean['Yr Qtr'].unique())}")
print(f"Avg quota attainment: {rp_clean['Quota Attainment %'].mean():.1f}%")
rp_clean.head()

In [ ]:
# Quick attainment distribution
rp_clean['Attainment Tier'].value_counts()

---
## 4. Sales Data
125 columns → keep 33 relevant ones · remove flagged duplicates · aggregate to rep × quarter

In [ ]:
sd = load(SALES_FILE, 'Sales Data')
sd.head(2)

In [ ]:
# ── 4a. Remove rows flagged as duplicates ─────────────────────────────────
dup_col = 'Is Duplicate Row?'
if dup_col in sd.columns:
    before = len(sd)
    sd = sd[sd[dup_col].astype(str).str.strip().str.upper() != 'TRUE']
    print(f"Dropped {before - len(sd):,} flagged duplicate rows → {len(sd):,} remaining")

In [ ]:
# ── 4b. Keep only useful columns (drop ~90 irrelevant/duplicate cols) ──────
SALES_KEEP = [
    # Join keys
    'Employee Number', 'Current Employee Number',
    'CustomerNumber', 'Current ParentNumber',
    'SalesOrderNumber', 'InvoiceNumber',
    # Time
    'SalesOrderDate', 'InvoiceDate', 'Invoice Yr-Qtr',
    # Revenue
    'NetSales', 'GrossSales', 'Quantity',
    # Product
    'Current ProductFamily', 'Finance Product Family',
    'Current ProductLine_Chr', 'ItemDescription', 'ItemNumber',
    'ItemType', 'Core20', 'Active Product',
    'FxItem', 'RxItem', 'BundledItem', 'Channel',
    # Customer
    'Current Territory', 'Current Division',
    'Current CustomerCategoryType', 'Current CustomerType',
    'Current CustomerLevel', 'Current BusinessUnit',
    'Current State', 'Current City',
    # Rep context
    'RepLevel', 'Rep Type',
    # Flags
    'OPP', 'Online Seller', 'Potential Diverter',
    'Account Classification', 'Clustered Account Type',
]

existing = [c for c in SALES_KEEP if c in sd.columns]
missing  = [c for c in SALES_KEEP if c not in sd.columns]
if missing:
    print(f"Note — these columns weren't found (check spelling): {missing}")

sd = sd[existing].copy()
print(f"Kept {len(existing)} columns")

In [ ]:
# ── 4c. Type cleanup ──────────────────────────────────────────────────────
sd['Employee Number']         = sd['Employee Number'].astype(str).str.strip()
sd['Current Employee Number'] = sd.get('Current Employee Number', sd['Employee Number']).astype(str).str.strip()
sd['SalesOrderDate']          = pd.to_datetime(sd.get('SalesOrderDate'), errors='coerce')
sd['InvoiceDate']             = pd.to_datetime(sd.get('InvoiceDate'),    errors='coerce')
sd['NetSales']                = pd.to_numeric(sd.get('NetSales'),  errors='coerce')
sd['GrossSales']              = pd.to_numeric(sd.get('GrossSales'), errors='coerce')
sd['Quantity']                = pd.to_numeric(sd.get('Quantity'),   errors='coerce')

if 'Invoice Yr-Qtr' in sd.columns:
    sd['Invoice Yr-Qtr'] = normalize_yr_qtr(sd['Invoice Yr-Qtr'])

sd.head(3)

In [ ]:
# ── 4d. Aggregate transactions → rep × quarter ────────────────────────────
sales_agg = (
    sd.groupby(['Employee Number', 'Invoice Yr-Qtr'])
    .agg(
        Txn_NetSales     = ('NetSales',        'sum'),
        Txn_GrossSales   = ('GrossSales',      'sum'),
        Txn_Quantity     = ('Quantity',         'sum'),
        Num_Orders       = ('SalesOrderNumber', 'nunique'),
        Num_Customers    = ('CustomerNumber',   'nunique'),
        Avg_Order_Value  = ('NetSales',         'mean'),
        Num_Rx_Items     = ('RxItem',  lambda x: (x.astype(str).str.upper() == 'TRUE').sum()),
        Num_Fx_Items     = ('FxItem',  lambda x: (x.astype(str).str.upper() == 'TRUE').sum()),
        Num_Core20_Items = ('Core20',  lambda x: (x.astype(str).str.upper() == 'TRUE').sum()),
    )
    .reset_index()
    .rename(columns={'Invoice Yr-Qtr': 'Yr Qtr'})
)

sales_agg['Avg_Order_Value'] = sales_agg['Avg_Order_Value'].round(2)
print(f"Aggregated: {len(sales_agg):,} employee-quarter rows from {len(sd):,} transactions")
sales_agg.head()

---
## 5. Merge — Build Master Dataset
**Spine:** Rep Performance (one row per rep per quarter)  
**Left joins:** Employee Info → Mileage → Sales Aggregate

In [ ]:
# Start with Rep Perf as the spine
master = rp_clean.copy()
print(f"Start (rep_perf):        {len(master):,} rows")

# + Employee Info (drop Position — already in rep_perf)
master = master.merge(
    emp_clean.drop(columns=['Position'], errors='ignore'),
    on='Employee Number', how='left'
)
print(f"After + Employee Info:   {len(master):,} rows")

# + Mileage (rep + quarter)
master = master.merge(mil_agg, on=['Employee Number', 'Yr Qtr'], how='left')
print(f"After + Mileage:         {len(master):,} rows  ({master['Total_Miles'].isna().sum()} quarters with no miles logged)")

# + Sales aggregate (rep + quarter)
master = master.merge(sales_agg, on=['Employee Number', 'Yr Qtr'], how='left')
print(f"After + Sales:           {len(master):,} rows  ({master['Txn_NetSales'].isna().sum()} quarters with no transactions)")

---
## 6. Final Calculated Fields

In [ ]:
# Efficiency: miles driven per $1,000 of revenue (lower = more efficient)
master['Miles per $1K Revenue'] = (
    master['Total_Miles'] / (master['Txn_NetSales'] / 1000)
).replace([np.inf, -np.inf], np.nan).round(2)

# Revenue per customer this quarter
master['Revenue per Customer'] = (
    master['Txn_NetSales'] / master['Num_Customers']
).replace([np.inf, -np.inf], np.nan).round(2)

# Cross-check: CRM quota sales vs actual invoiced transactions
# A big gap can flag returns, adjustments, or data mismatches
master['Sales vs Txn Diff $'] = (master['Sales'] - master['Txn_NetSales']).round(2)
master['Sales vs Txn Diff %'] = (
    master['Sales vs Txn Diff $'] / master['Sales'] * 100
).replace([np.inf, -np.inf], np.nan).round(2)

print('Calculated fields added')

---
## 7. Column Order & Final Preview

In [ ]:
COL_ORDER = [
    # Rep identity
    'Employee Number', 'Position', 'Status Classification', 'Is Active',
    'Original Hire Date', 'Termination Date', 'Tenure Years',
    # Period & territory
    'Yr Qtr', 'Division', 'DivisionName', 'Territory', 'TerritoryName',
    # Quota & attainment
    'Goal', 'Sales', 'Quota Attainment %', 'Gap to Quota $', 'Attainment Tier', 'Hit Quota',
    # Transaction sales
    'Txn_NetSales', 'Txn_GrossSales', 'Txn_Quantity',
    'Num_Orders', 'Num_Customers', 'Avg_Order_Value',
    'Num_Rx_Items', 'Num_Fx_Items', 'Num_Core20_Items',
    # Cross-check
    'Sales vs Txn Diff $', 'Sales vs Txn Diff %',
    # Efficiency
    'Revenue per Customer', 'Total_Miles', 'Mileage_Trips', 'Miles per $1K Revenue',
]

extra = [c for c in master.columns if c not in COL_ORDER]
final_order = [c for c in COL_ORDER + extra if c in master.columns]
master = master[final_order]

print(f"Final shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
master.head(10)

---
## 8. Quick QA Checks

In [ ]:
print('=== Summary Stats ===')
print(f"Unique reps      : {master['Employee Number'].nunique():,}")
print(f"Quarters covered : {sorted(master['Yr Qtr'].unique())}")
print(f"Active reps      : {master['Is Active'].sum()}")
print(f"Avg attainment   : {master['Quota Attainment %'].mean():.1f}%")
print(f"Reps at quota    : {master['Hit Quota'].sum()} / {len(master)}")
print(f"\nMissing values per key column:")
key_cols = ['Goal', 'Sales', 'Quota Attainment %', 'Txn_NetSales', 'Total_Miles', 'Is Active']
print(master[key_cols].isna().sum())

In [ ]:
# Attainment breakdown
print('=== Attainment Tier Distribution ===')
master['Attainment Tier'].value_counts()

In [ ]:
# Top 10 reps by quota attainment this period
print('=== Top 10 Reps by Quota Attainment % ===')
(
    master[['Employee Number', 'Yr Qtr', 'TerritoryName', 'Goal', 'Sales', 'Quota Attainment %', 'Attainment Tier']]
    .sort_values('Quota Attainment %', ascending=False)
    .head(10)
)

In [ ]:
# Largest Sales vs Transaction mismatch (potential data issues)
print('=== Largest CRM vs Transaction Gaps (top 10) ===')
(
    master[['Employee Number', 'Yr Qtr', 'Sales', 'Txn_NetSales', 'Sales vs Txn Diff $', 'Sales vs Txn Diff %']]
    .dropna(subset=['Sales vs Txn Diff $'])
    .reindex(master['Sales vs Txn Diff $'].abs().nlargest(10).index)
)

---
## 9. Export

In [ ]:
master.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')

master.to_excel(OUTPUT_XLSX, index=False, engine='openpyxl')
print(f'Saved: {OUTPUT_XLSX}')

print(f'\nDone. {master.shape[0]:,} rows × {master.shape[1]} columns exported.')